In [1]:
import anndata as ad
import numpy as np
import pandas as pd
import zarr

print("anndata:", ad.__version__)
print("pandas:", pd.__version__)
print("zarr:", zarr.__version__)


obs = pd.DataFrame(
    {
        "np_nan_str": [np.nan, "cell1"],
        "np_nan_int": [np.nan, 1],
        "np_nan_float": [np.nan, 1.0],
        "pd_NA_str": [pd.NA, "cell1"],
        "pd_NA_int": [pd.NA, 1],
        "pd_NA_float": [pd.NA, 1.0],
    },
    index=["cell1", "cell2"],
)
print(obs.dtypes)

adata = ad.AnnData(X=np.zeros((2, 1)), obs=obs, var=pd.DataFrame(index=["gene1"]), uns={"test": obs.copy()})

print("Obs Before writing")
print(adata.obs)
print(adata.obs.isna())

uns_test = adata.uns["test"]
print("Uns Before writing")
print(uns_test)
print(uns_test.isna())

adata.write_zarr("toy_anndata.zarr")

reloaded = ad.read_zarr("toy_anndata.zarr")

print("\nObs After reading")
print(reloaded.obs)
print(reloaded.obs.isna())

print("\nObs Dtypes")
print(reloaded.obs.dtypes)

reloaded_uns = reloaded.uns["test"]
print("\nUns After reading")
print(reloaded_uns)
print(reloaded_uns.isna())

print("\nUns Dtypes")
print(reloaded_uns.dtypes)

/tmp/ipykernel_39917/3882263005.py:6: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)


anndata: 0.13.2
pandas: 2.3.3
zarr: 3.2.1
np_nan_str       object
np_nan_int      float64
np_nan_float    float64
pd_NA_str        object
pd_NA_int        object
pd_NA_float      object
dtype: object
Obs Before writing
      np_nan_str  np_nan_int  np_nan_float pd_NA_str pd_NA_int pd_NA_float
cell1        NaN         NaN           NaN      <NA>      <NA>        <NA>
cell2      cell1         1.0           1.0     cell1         1         1.0
       np_nan_str  np_nan_int  np_nan_float  pd_NA_str  pd_NA_int  pd_NA_float
cell1        True        True          True       True       True         True
cell2       False       False         False      False      False        False
Uns Before writing
      np_nan_str  np_nan_int  np_nan_float pd_NA_str pd_NA_int pd_NA_float
cell1        NaN         NaN           NaN      <NA>      <NA>        <NA>
cell2      cell1         1.0           1.0     cell1         1         1.0
       np_nan_str  np_nan_int  np_nan_float  pd_NA_str  pd_NA_int  pd_NA_fl

In [2]:
import anndata as ad
import geopandas as gpd
import numpy as np
import pandas as pd
import spatialdata as sd
import zarr
from shapely.geometry import Polygon
from spatialdata.models import ShapesModel, TableModel

print("spatialdata:", sd.__version__)
print("anndata:", ad.__version__)
print("pandas:", pd.__version__)
print("zarr:", zarr.__version__)

shapes = gpd.GeoDataFrame(
    {
        "geometry": [
            Polygon([(0, 0), (1, 0), (1, 1), (0, 1)]),
            Polygon([(2, 0), (3, 0), (3, 1), (2, 1)]),
        ]
    },
    index=["cell1", "cell2"],
)

obs = pd.DataFrame(
    {
        "instance_id": ["cell1", "cell2"],
        "region": ["cells", "cells"],
        "np_nan_str": [np.nan, "cell1"],
        "np_nan_int": [np.nan, 1],
        "np_nan_float": [np.nan, 1.0],
        "pd_NA_str": [pd.NA, "cell1"],
        "pd_NA_int": [pd.NA, 1],
        "pd_NA_float": [pd.NA, 1.0],
    },
    index=["cell1", "cell2"],
)

adata = ad.AnnData(X=np.zeros((2, 1)), obs=obs, var=pd.DataFrame(index=["gene1"]), uns={"test": obs.copy()})

table = TableModel.parse(
    adata,
    region="cells",
    region_key="region",
    instance_key="instance_id",
)

sdata = sd.SpatialData(
    shapes={"cells": ShapesModel.parse(shapes)},
    tables={"counts": table},
)

print("\nObs Before writing")
print(sdata.tables["counts"].obs)
print("\nObs Missing values before writing")
print(sdata.tables["counts"].obs.isna())
print("\nObs Dtypes before writing")
print(sdata.tables["counts"].obs.dtypes)

uns_test = sdata.tables["counts"].uns["test"]
print("\nUns Before writing")
print(uns_test)
print("\nUns Missing values before writing")
print(uns_test.isna())
print("\nUns Dtypes before writing")
print(uns_test.dtypes)

sdata.write("toy_spatialdata.zarr", overwrite=True)

reloaded = sd.read_zarr("toy_spatialdata.zarr")

print("\nObs After reading")
print(reloaded.tables["counts"].obs)
print(reloaded.tables["counts"].obs.isna())
print("\nObs Dtypes after reading")
print(reloaded.tables["counts"].obs.dtypes)

reloaded_uns = reloaded.tables["counts"].uns["test"]
print("\nUns After reading")
print(reloaded_uns)
print("\nUns Missing values after reading")
print(reloaded_uns.isna())
print("\nUns Dtypes after reading")
print(reloaded_uns.dtypes)

spatialdata: 0.8.0
anndata: 0.13.2
pandas: 2.3.3
zarr: 3.2.1


/tmp/ipykernel_39917/2053751917.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)
/workdata/repos/scverse/spatialdata-family/spatialdata/src/spatialdata/models/models.py:1267: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)



Obs Before writing
      instance_id region np_nan_str  np_nan_int  np_nan_float pd_NA_str  \
cell1       cell1  cells        NaN         NaN           NaN      <NA>   
cell2       cell2  cells      cell1         1.0           1.0     cell1   

      pd_NA_int pd_NA_float  
cell1      <NA>        <NA>  
cell2         1         1.0  

Obs Missing values before writing
       instance_id  region  np_nan_str  np_nan_int  np_nan_float  pd_NA_str  \
cell1        False   False        True        True          True       True   
cell2        False   False       False       False         False      False   

       pd_NA_int  pd_NA_float  
cell1       True         True  
cell2      False        False  

Obs Dtypes before writing
instance_id       object
region          category
np_nan_str        object
np_nan_int       float64
np_nan_float     float64
pd_NA_str         object
pd_NA_int         object
pd_NA_float       object
dtype: object

Uns Before writing
      instance_id region np_nan_st